# Live demo: searching 511,958 company documents

**Before the audience arrives:** run every cell in **Part 1** (Runtime → Run all works).
The first session takes about 10–15 minutes because it builds the search index and
saves it to Google Drive. Later sessions reuse it and are ready in a few minutes.

**During the demo:** use Part 2.

## Part 1: Setup (run before the demo)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
WORK = "/content/drive/MyDrive/rag-dense"      # Dataset and indexes are kept here.
CORPUS = f"{WORK}/documents.parquet"
QUESTIONS_FILE = f"{WORK}/questions.jsonl"
DRIVE_INDEX = f"{WORK}/indexes/bm25"
INDEX = "/content/indexes/bm25"               # Local copy for fast searching.
PROJECT = "/content/project"

In [ ]:
BRANCH = "main"
if not os.path.exists(PROJECT):
    !git clone --depth 1 -b {BRANCH} https://github.com/hira-mian/Agentic-Enterprise-RAG-System.git {PROJECT}
%cd {PROJECT}
!pip install -q -r requirements.txt

Build the dataset if it is not on Drive yet (about 5–10 minutes), then build the
BM25+ index once (about 5 minutes) and save it to Drive. Later sessions just copy it.

In [ ]:
if not os.path.exists(CORPUS):
    !python -m src.data.corpus --clone --output {CORPUS}

if not os.path.exists(INDEX):
    if os.path.exists(DRIVE_INDEX):
        print("Copying the saved index from Drive...")
        !mkdir -p /content/indexes && cp -r {DRIVE_INDEX} {INDEX}
    else:
        print("Building the index (about 5 minutes)...")
        !python -m src.retrieval.corpus_bm25 --corpus {CORPUS} --output {INDEX}
        !mkdir -p {WORK}/indexes && cp -r {INDEX} {DRIVE_INDEX}
print("Index ready.")

Load everything into memory once (about 1–2 minutes). After this, every search is instant.

In [ ]:
import json
import sys
import time
from pathlib import Path

import pyarrow.parquet as pq

sys.path.insert(0, PROJECT)
from src.retrieval.corpus_bm25 import CorpusBM25

started = time.time()
bm25 = CorpusBM25.load(Path(INDEX), mmap=False)  # Load into memory so queries are instant.
docs = pq.read_table(CORPUS, columns=["doc_id", "source_type", "title"])
INFO = dict(zip(docs.column("doc_id").to_pylist(),
                zip(docs.column("source_type").to_pylist(), docs.column("title").to_pylist())))
QUESTIONS = {q["question_id"]: q for q in map(json.loads, open(QUESTIONS_FILE))}
dense = None  # Set in the optional BGE section.
print(f"Ready: {len(bm25.doc_ids):,} documents loaded in {time.time() - started:.0f}s")


def show(hits, expected=(), label="BM25+", elapsed=None):
    print(f"--- {label} ---")
    for rank, (doc_id, _) in enumerate(hits, 1):
        source, title = INFO[doc_id]
        mark = "   <-- correct" if doc_id in expected else ""
        print(f"{rank:>2}. [{source}] {title[:70]}{mark}")
    if elapsed is not None:
        print(f"(searched {len(bm25.doc_ids):,} documents in {elapsed * 1000:.0f} ms)")
    if expected:
        found = len(set(expected) & {d for d, _ in hits})
        print(f"Correct documents in top {len(hits)}: {found} of {len(expected)}")


def ask(question, top_k=10, expected=()):
    """Search all documents for any question you type."""
    print(f"Q: {question}\n")
    start = time.perf_counter()
    hits = bm25.rank(question, top_k)
    show(hits, expected, "BM25+ (keyword search)", time.perf_counter() - start)
    if dense is not None:
        print()
        start = time.perf_counter()
        hits = dense.rank(question, top_k)
        show(hits, expected, "BGE-small (meaning search)", time.perf_counter() - start)


def ask_benchmark(question_id, top_k=10):
    """Run one benchmark question and mark its correct documents."""
    q = QUESTIONS[question_id]
    print(f"[{question_id} | type: {q['question_type']}]")
    ask(q["question"], top_k, set(q["expected_doc_ids"]))
    if q["expected_doc_ids"]:
        print("\nThe correct document(s):")
        for doc_id in q["expected_doc_ids"]:
            source, title = INFO[doc_id]
            print(f"  [{source}] {title[:70]}")
    print(f"\nReference answer: {q['gold_answer']}")

## Part 2: The demo

### 1. The code is tested
38 automated tests check the dataset build, both search methods, the metrics, and the
answer generator. They also run on GitHub on every push.

In [ ]:
!python -m pytest -q

### 2. Run the full benchmark live
Searches all 511,958 documents for the 300 development questions and scores the results.

In [ ]:
!python -m src.evaluation.run_retrieval --method bm25 --split development --index {INDEX} --questions {QUESTIONS_FILE} > /dev/null
s = json.load(open(f"{PROJECT}/evaluation/results/bm25_full_development/summary.json"))
o, t = s["overall"], s["timing_seconds"]
print(f"{s['questions']} questions ({o['recall@10']['scored']} scored) searched in {t['total_queries']:.1f} s")
for metric in ("recall@5", "recall@10", "recall@20", "ndcg@10"):
    print(f"{metric:>10}: {o[metric]['mean']:.3f}")

### 3. Ask benchmark questions
Each benchmark question has known correct documents, marked with `<-- correct`.

**Keyword search at its best:** the question names the customer "NorthPoint Signalworks".

In [ ]:
ask_benchmark("qst_0063", top_k=5)

**Hard case:** the correct pull request is surrounded by very similar ones.

In [ ]:
ask_benchmark("qst_0001", top_k=5)

**Failure:** a reworded question that never uses the document's own words.

In [ ]:
ask_benchmark("qst_0178", top_k=5)

### 4. Ask your own question

In [ ]:
ask("Who is responsible for signing off on latency regressions for Nimbus Health?", top_k=5)

### 5. Results
Scores on the 300 development questions (282 have correct documents to find).

In [ ]:
def load_summary(name):
    return json.load(open(f"{PROJECT}/evaluation/results/{name}_full_development/summary.json"))

bm, bge = load_summary("bm25"), load_summary("dense")
rows = [("Overall", bm["overall"], bge["overall"])] + [
    (t, bm["question_types"][t], bge["question_types"][t])
    for t in bm["question_types"] if bm["question_types"][t]["recall@10"]["mean"] is not None
]
print(f"{'Question type':26} {'Questions':>9} {'BM25+ R@10':>11} {'BGE R@10':>9}")
for name, a, b in rows:
    print(f"{name:26} {a['recall@10']['scored']:>9} {a['recall@10']['mean']:>11.3f} {b['recall@10']['mean']:>9.3f}")

## Optional: compare with BGE (meaning search) live
Needs a **GPU runtime** (Runtime → Change runtime type → T4 GPU) and the BGE embeddings
on Drive from the baseline notebook. Loading takes a few minutes. After this, `ask` and
`ask_benchmark` show both methods side by side.

In [ ]:
import torch

if torch.cuda.is_available():
    from src.retrieval.corpus_dense import BGEModel, CorpusDense

    dense = CorpusDense.load(Path(f"{WORK}/dense"), BGEModel("cuda", fp16=True))
    print(f"BGE ready: {dense.index.ntotal:,} passages")
else:
    print("No GPU: skipping BGE. Switch to a T4 GPU runtime to enable it.")

BGE wins on this one: the question describes the topic without the document's exact wording.

In [ ]:
ask_benchmark("qst_0417", top_k=5)